# GIADA Task 17e — conferma causale indipendente di Ca_HVA
Leggi `experiments/task17e_independent_causal_confirmation_preregistration.md` prima del run. Questo notebook testa Gate C sui seed nuovi e scarica un archivio diagnostico anche quando un controllo tecnico fallisce. Non richiede dataset Kaggle aggiuntivi.

In [ ]:
from pathlib import Path
import base64,json,shutil,subprocess,sys,tempfile,uuid
from IPython.display import Javascript,display
WORK=Path('/kaggle/working')/f'giada_task17e_{uuid.uuid4().hex[:8]}'
REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net';WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert (REPO/'src/giada_teacher/roadmap_task17e_independent_causal_confirmation.py').is_file()
assert (REPO/'experiments/task17e_independent_causal_confirmation_preregistration.md').is_file()
print({'giada_revision':REVISION,'teacher_revision':'074c4666300a8ad246601dab179a97a6942f0f29'})

## Preflight NEURON
Usa notebook, codice e protocollo dello stesso commit. Questa è una simulazione NEURON su CPU; la GPU non serve.

In [ ]:
try:
 import neuron
except ModuleNotFoundError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__.split('+',1)[0]=='8.2.7',f'NEURON {neuron.__version__} non corrisponde a 8.2.7'
nrnivmodl=shutil.which('nrnivmodl') or str(Path(sys.executable).parent/'nrnivmodl')
assert Path(nrnivmodl).is_file() and shutil.which('gcc'),'Compilatore NEURON/gcc assente'
simulation_dir=TEACHER/'L5PC_NEURON_simulation'
if not list(simulation_dir.rglob('libnrnmech.so')):subprocess.run([nrnivmodl,'mods'],cwd=simulation_dir,check=True)
assert list(simulation_dir.rglob('libnrnmech.so')),'Meccanismi canonici non compilati'
print({'neuron':neuron.__version__,'canonical_mechanisms':'ready'})

## Matrice congelata e Gate C
Il runner esegue 135 episodi, stampa soltanto progressi sintetici e salva report dettagliati. Se una precondizione fallisce, lascia aperta la possibilità di scaricare i log.

In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/f'giada_task17e_causal_confirmation_{REVISION[:7]}_{uuid.uuid4().hex[:8]}'
command=[sys.executable,'-u',str(REPO/'scripts/run_roadmap_task17e.py'),'--repo',str(REPO),'--teacher',str(TEACHER),'--output',str(OUTPUT)]
completed=subprocess.run(command,check=False)
path=OUTPUT/('failure_report.json' if (OUTPUT/'failure_report.json').is_file() else 'final_report.json')
report=json.loads(path.read_text()) if path.is_file() else {'valid':False,'decision':'SUPERVISOR_INTERRUPTED','error':f'exit={completed.returncode}; controllare process.log'}
display({key:report.get(key) for key in ('valid','decision','gate_c_authorized','task18_authorized','gates','event_coverage','episode_count','comparison_count','worst_candidate_voltage_rmse_mv','worst_candidate_gate_error','worst_candidate_current_rmse_ma_cm2','worst_identifiable_effect_relative_error','runtime_total_seconds_by_arm','error_type','error')})
print({'output':str(OUTPUT),'returncode':completed.returncode,'next':'Scarica lo ZIP anche se il processo fallisce'})

## Download del risultato
Esporta report, metriche, eventi, tracciati ridotti e log. Gli array completi non vengono stampati nella pagina Kaggle.

In [ ]:
EXPORT=Path(tempfile.mkdtemp(prefix='task17e_download_',dir=WORK))
for name in ('final_report.json','failure_report.json','process_status.json','process.log','last_phase.json','paired_metrics.json','paired_effects.json','runtime_seconds.json','diagnostic_traces_1ms.json'):
 source=OUTPUT/name
 if source.is_file():shutil.copy2(source,EXPORT/name)
archive=Path(shutil.make_archive(str(Path('/kaggle/working')/OUTPUT.name),'zip',EXPORT.parent,EXPORT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})